# 01 — Analyse Exploratoire (EDA)

**Projet : ObRail — Détection des sous-dessertes ferroviaires**

**Objectif** : explorer le jeu de données enrichi pour comprendre sa structure, valider sa qualité, et **vérifier si les features permettent de prédire la cible `is_underserved`** (ligne sous-desservie : demande > offre) *avant* la phase de modélisation.

- **Input** : `../data/processed/routes_processed.csv`
- **Outputs** : observations documentées + figures dans `../evaluation/plots/`
- **Type de problème** : classification binaire (`is_underserved` ∈ {0,1}), classes déséquilibrées
- **Date** : 2026

##  — Setup & imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

RANDOM_STATE = 42
pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')

PLOTS_DIR = Path('../evaluation/plots')
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv('../data/processed/routes_processed.csv')
print(f"Shape : {df.shape}")
print(f"\nColonnes : {df.columns.tolist()}")
print(f"\nAperçu :")
df.head()

## 1 - Qualité des données

In [ ]:
print("=== Valeurs manquantes ===")
missing = df.isnull().sum()
print(missing[missing > 0] if len(missing[missing > 0]) else "✅ Aucun missing")

print(f"\n=== Doublons : {df.duplicated().sum()} ===")

print("\n=== Types ===")
print(df.dtypes)

print("\n=== Statistiques numériques ===")
print(df[['trip_count', 'log_trip_count']].describe().round(2))

## 2 - Analyse de la cible is_underserved

In [ ]:
counts = df['is_underserved'].value_counts()
pct    = df['is_underserved'].value_counts(normalize=True).mul(100).round(1)

print(pd.DataFrame({'effectif': counts, 'part_%': pct}))

fig, ax = plt.subplots(figsize=(5, 4))
sns.barplot(x=counts.index, y=counts.values, hue=counts.index,
            palette=['#4c72b0', '#dd8452'], legend=False, ax=ax)
ax.set_title("Distribution de la cible is_underserved")
ax.set_xlabel('is_underserved')
ax.set_ylabel('Nombre de routes')
for i, v in enumerate(counts.values):
    ax.text(i, v + 10, f'{v}\n({pct.iloc[i]}%)', ha='center')
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'eda_target_balance.png', dpi=110)
plt.show()

## 3 - Répartition jour vs nuit

In [ ]:
print("Taux de sous-desserte par type de service :")
print(df.groupby('service_type')['is_underserved']
        .value_counts(normalize=True).mul(100).round(1))

print("\nNombre de routes par type :")
print(df.groupby(['service_type', 'is_underserved']).size().unstack())

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, col in zip(axes, ['service_type', 'country']):
    rate = df.groupby(col)['is_underserved'].mean().mul(100).sort_values()
    sns.barplot(x=rate.index, y=rate.values, ax=ax, color='#dd8452')
    ax.set_title(f'Taux de sous-desserte par {col}')
    ax.set_ylabel('%')
    ax.set_ylim(0, 60)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'eda_underserved_by_group.png', dpi=110)
plt.show()

## 4 - Distribution du trip_count

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Distribution brute (log scale)
sns.histplot(df['trip_count'], bins=50, ax=axes[0], color='#4c72b0')
axes[0].set_title('Distribution trip_count (brute)')
axes[0].set_xlabel('Nombre de trips')

# Distribution log
sns.histplot(df['log_trip_count'], bins=50, ax=axes[1], color='#4c72b0')
axes[1].set_title('Distribution log_trip_count')
axes[1].set_xlabel('log(trip_count)')

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'eda_distributions.png', dpi=110)
plt.show()

# Boxplot trip_count par classe
fig, ax = plt.subplots(figsize=(6, 4))
sns.boxplot(data=df, x='is_underserved', y='log_trip_count',
            hue='is_underserved', palette=['#4c72b0', '#dd8452'],
            legend=False, ax=ax)
ax.set_title('log_trip_count par classe is_underserved')
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'eda_features_vs_target.png', dpi=110)
plt.show()

## 5 - Analyse géographique

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
geo = df.groupby('country')['is_underserved'].mean().mul(100).sort_values(ascending=False)
sns.barplot(x=geo.index, y=geo.values, ax=ax, color='#dd8452')
ax.axhline(df['is_underserved'].mean() * 100, ls='--', color='grey', label='taux global')
ax.set_title('Taux de sous-desserte par pays')
ax.set_ylabel('%')
ax.legend()
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'eda_underserved_by_country.png', dpi=110)
plt.show()

print("Taux de sous-desserte par pays :")
print(df.groupby('country')['is_underserved'].mean().mul(100).round(1).sort_values(ascending=False))
print("\nVolume par pays :")
print(df['country'].value_counts())

## 6 — Synthèse

In [ ]:
print("""
=== SYNTHÈSE EDA ===

Qualité : 2440 lignes, 0 doublons, données GTFS réelles.

Cible : is_underserved binaire — 66.5% non sous-desservies, 33.5% sous-desservies.
Classe déséquilibrée mais acceptable — pas besoin de SMOTE.

Findings clés :
- Les trains de nuit sont plus sous-desservis (37%) que les trains de jour (26.2%)
- La fréquence (trip_count) est le signal principal — bien séparée entre classes
- log_trip_count réduit l'asymétrie forte du trip_count brut

Implications pour la modélisation :
- Utiliser log_trip_count plutôt que trip_count brut
- Évaluer sur F1/recall classe 1, ROC-AUC — pas l'accuracy
- Features principales : log_trip_count, service_type, country, is_international
""")